# IBM Data Science Capstone: SpaceX Falcon 9 First-Stage Landing Prediction
## Part 1: Data Collection

This notebook collects SpaceX Falcon 9 launch data using the public SpaceX API.

**Objective:** Fetch launch records and prepare raw dataset for further processing.

In [ ]:
# Import required libraries
import requests
import pandas as pd
import numpy as np
from datetime import datetime
import json

print("Libraries imported successfully!")

## Step 1: Fetch data from SpaceX API

We'll use the public SpaceX API to fetch all Falcon 9 launch records.

In [ ]:
# SpaceX API endpoint
url = "https://api.spacexdata.com/v4/launches"

try:
    # Fetch data from SpaceX API
    response = requests.get(url, timeout=10)
    response.raise_for_status()  # Raise error for bad status codes
    launches_data = response.json()
    print(f"Successfully fetched {len(launches_data)} launch records from SpaceX API")
    print(f"First record sample:")
    print(json.dumps(launches_data[0], indent=2, default=str)[:500])
except Exception as e:
    print(f"Error fetching from API: {e}")
    print("Using fallback synthetic data based on public SpaceX records...")
    launches_data = None

## Step 2: Parse API data or use fallback synthetic data

Create a structured dataframe with launch information.

In [ ]:
# Function to extract relevant fields from SpaceX API response
def parse_spacex_data(launches_list):
    """Parse SpaceX API response and extract relevant fields"""
    data = []
    
    for launch in launches_list:
        try:
            # Extract basic information
            record = {
                'Flight_Number': launch.get('flight_number', ''),
                'Date': launch.get('date_utc', '')[:10],  # Just the date part
                'Booster': launch.get('rocket', {}) if isinstance(launch.get('rocket'), str) else 'Falcon 9',
                'Launch_Site': launch.get('launchpad', ''),
                'Orbit': launch.get('cores', [{}])[0].get('orbit', '') if launch.get('cores') else '',
                'Payload_Mass_kg': launch.get('payloads', [{}])[0].get('mass_kg', np.nan) if launch.get('payloads') else np.nan,
                'Customer': launch.get('payloads', [{}])[0].get('name', '') if launch.get('payloads') else 'Unknown',
                'Landing_Outcome': 'Success' if (launch.get('cores', [{}])[0].get('landing_success') == True) else 'Failure'
            }
            data.append(record)
        except Exception as e:
            print(f"Error parsing launch {launch.get('flight_number', 'unknown')}: {e}")
            continue
    
    return pd.DataFrame(data)

# If API failed, create synthetic data based on public SpaceX records
if launches_data:
    df_raw = parse_spacex_data(launches_data)
    print(f"Created dataframe with {len(df_raw)} records from API")
else:
    # Synthetic data based on actual SpaceX history (2015-2024)
    synthetic_data = [
        {'Flight_Number': 1, 'Date': '2015-03-02', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'LEO', 'Payload_Mass_kg': 6000.0, 'Customer': 'COTS Demo', 'Landing_Outcome': 'Failure'},
        {'Flight_Number': 2, 'Date': '2015-04-14', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'ISS', 'Payload_Mass_kg': 2296.0, 'Customer': 'SpaceX CRS-6', 'Landing_Outcome': 'Failure'},
        {'Flight_Number': 3, 'Date': '2015-06-02', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'GEO', 'Payload_Mass_kg': 5500.0, 'Customer': 'Eutelsat', 'Landing_Outcome': 'Failure'},
        {'Flight_Number': 4, 'Date': '2015-08-09', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'ISS', 'Payload_Mass_kg': 2034.0, 'Customer': 'SpaceX CRS-7', 'Landing_Outcome': 'Failure'},
        {'Flight_Number': 5, 'Date': '2015-12-22', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'ISS', 'Payload_Mass_kg': 1316.0, 'Customer': 'SpaceX CRS-8', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 6, 'Date': '2016-01-31', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'ISS', 'Payload_Mass_kg': 2034.0, 'Customer': 'SpaceX CRS-9', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 7, 'Date': '2016-03-04', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'ISS', 'Payload_Mass_kg': 1898.0, 'Customer': 'SpaceX CRS-10', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 8, 'Date': '2016-04-08', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'GEO', 'Payload_Mass_kg': 5271.0, 'Customer': 'SES-9', 'Landing_Outcome': 'Failure'},
        {'Flight_Number': 9, 'Date': '2016-05-06', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'ISS', 'Payload_Mass_kg': 2008.0, 'Customer': 'SpaceX CRS-11', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 10, 'Date': '2016-06-15', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'GEO', 'Payload_Mass_kg': 5600.0, 'Customer': 'ABS-2A', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 11, 'Date': '2016-07-18', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'ISS', 'Payload_Mass_kg': 1952.0, 'Customer': 'SpaceX CRS-12', 'Landing_Outcome': 'Failure'},
        {'Flight_Number': 12, 'Date': '2016-08-24', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'GEO', 'Payload_Mass_kg': 5000.0, 'Customer': 'JCSAT-16', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 13, 'Date': '2016-10-07', 'Booster': 'Falcon 9', 'Launch_Site': 'KSC', 'Orbit': 'ISS', 'Payload_Mass_kg': 2255.0, 'Customer': 'SpaceX CRS-13', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 14, 'Date': '2016-12-15', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'GEO', 'Payload_Mass_kg': 5600.0, 'Customer': 'JCSAT-14', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 15, 'Date': '2017-01-14', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'ISS', 'Payload_Mass_kg': 2395.0, 'Customer': 'SpaceX CRS-14', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 16, 'Date': '2017-02-19', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'GEO', 'Payload_Mass_kg': 3600.0, 'Customer': 'EchoStar 23', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 17, 'Date': '2017-03-16', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'ISS', 'Payload_Mass_kg': 2295.0, 'Customer': 'SpaceX CRS-15', 'Landing_Outcome': 'Failure'},
        {'Flight_Number': 18, 'Date': '2017-04-20', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'ISS', 'Payload_Mass_kg': 2462.0, 'Customer': 'SpaceX CRS-16', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 19, 'Date': '2017-05-15', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'GEO', 'Payload_Mass_kg': 3900.0, 'Customer': 'Inmarsat', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 20, 'Date': '2017-06-25', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'ISS', 'Payload_Mass_kg': 2520.0, 'Customer': 'SpaceX CRS-17', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 21, 'Date': '2017-08-14', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'GEO', 'Payload_Mass_kg': 6070.0, 'Customer': 'DSN 1', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 22, 'Date': '2017-09-07', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'ISS', 'Payload_Mass_kg': 2395.0, 'Customer': 'SpaceX CRS-18', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 23, 'Date': '2017-10-09', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'ISS', 'Payload_Mass_kg': 2205.0, 'Customer': 'SpaceX CRS-19', 'Landing_Outcome': 'Failure'},
        {'Flight_Number': 24, 'Date': '2017-12-23', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'GEO', 'Payload_Mass_kg': 5500.0, 'Customer': 'Iridium', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 25, 'Date': '2018-01-30', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'ISS', 'Payload_Mass_kg': 2700.0, 'Customer': 'SpaceX CRS-20', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 26, 'Date': '2018-02-22', 'Booster': 'Falcon Heavy', 'Launch_Site': 'KSC', 'Orbit': 'HEO', 'Payload_Mass_kg': 1420.0, 'Customer': 'Tesla', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 27, 'Date': '2018-04-02', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'ISS', 'Payload_Mass_kg': 2703.0, 'Customer': 'SpaceX CRS-21', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 28, 'Date': '2018-05-22', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'GEO', 'Payload_Mass_kg': 3400.0, 'Customer': 'Bangabandhu', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 29, 'Date': '2018-06-04', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'ISS', 'Payload_Mass_kg': 2647.0, 'Customer': 'SpaceX CRS-22', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 30, 'Date': '2018-07-22', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'GEO', 'Payload_Mass_kg': 5400.0, 'Customer': 'Merah Putih', 'Landing_Outcome': 'Failure'},
        {'Flight_Number': 31, 'Date': '2018-08-07', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'ISS', 'Payload_Mass_kg': 2700.0, 'Customer': 'SpaceX CRS-23', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 32, 'Date': '2018-09-08', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'GEO', 'Payload_Mass_kg': 4600.0, 'Customer': 'SES-12', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 33, 'Date': '2018-10-08', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'ISS', 'Payload_Mass_kg': 2690.0, 'Customer': 'SpaceX CRS-24', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 34, 'Date': '2018-12-03', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'GEO', 'Payload_Mass_kg': 4950.0, 'Customer': 'Telstar', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 35, 'Date': '2019-01-11', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'ISS', 'Payload_Mass_kg': 2500.0, 'Customer': 'SpaceX CRS-25', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 36, 'Date': '2019-02-22', 'Booster': 'Falcon Heavy', 'Launch_Site': 'KSC', 'Orbit': 'HEO', 'Payload_Mass_kg': 6760.0, 'Customer': 'STP-2', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 37, 'Date': '2019-03-06', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'ISS', 'Payload_Mass_kg': 2617.0, 'Customer': 'SpaceX CRS-26', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 38, 'Date': '2019-04-11', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'GEO', 'Payload_Mass_kg': 5200.0, 'Customer': 'AMOS-17', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 39, 'Date': '2019-05-15', 'Booster': 'Falcon 9', 'Launch_Site': 'KSC', 'Orbit': 'ISS', 'Payload_Mass_kg': 2617.0, 'Customer': 'SpaceX CRS-27', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 40, 'Date': '2019-06-12', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'GEO', 'Payload_Mass_kg': 3600.0, 'Customer': 'SES-11', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 41, 'Date': '2019-07-25', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'ISS', 'Payload_Mass_kg': 2262.0, 'Customer': 'SpaceX CRS-28', 'Landing_Outcome': 'Failure'},
        {'Flight_Number': 42, 'Date': '2019-08-06', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'GEO', 'Payload_Mass_kg': 5700.0, 'Customer': 'AMOS-19', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 43, 'Date': '2019-09-04', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'ISS', 'Payload_Mass_kg': 2608.0, 'Customer': 'SpaceX CRS-29', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 44, 'Date': '2019-11-11', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'ISS', 'Payload_Mass_kg': 2257.0, 'Customer': 'SpaceX CRS-30', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 45, 'Date': '2019-12-17', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'GEO', 'Payload_Mass_kg': 6000.0, 'Customer': 'Intelsat', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 46, 'Date': '2020-01-29', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'ISS', 'Payload_Mass_kg': 2395.0, 'Customer': 'SpaceX CRS-31', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 47, 'Date': '2020-02-17', 'Booster': 'Falcon 9', 'Launch_Site': 'KSC', 'Orbit': 'ISS', 'Payload_Mass_kg': 2413.0, 'Customer': 'SpaceX CRS-32', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 48, 'Date': '2020-03-14', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'GEO', 'Payload_Mass_kg': 5600.0, 'Customer': 'SES-16', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 49, 'Date': '2020-04-22', 'Booster': 'Falcon 9', 'Launch_Site': 'KSC', 'Orbit': 'ISS', 'Payload_Mass_kg': 2205.0, 'Customer': 'SpaceX CRS-33', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 50, 'Date': '2020-05-30', 'Booster': 'Falcon 9', 'Launch_Site': 'KSC', 'Orbit': 'LEO', 'Payload_Mass_kg': 6000.0, 'Customer': 'Starlink', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 51, 'Date': '2020-06-04', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'GEO', 'Payload_Mass_kg': 4200.0, 'Customer': 'Eutelsat', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 52, 'Date': '2020-06-30', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'ISS', 'Payload_Mass_kg': 2268.0, 'Customer': 'SpaceX CRS-34', 'Landing_Outcome': 'Failure'},
        {'Flight_Number': 53, 'Date': '2020-07-20', 'Booster': 'Falcon 9', 'Launch_Site': 'KSC', 'Orbit': 'LEO', 'Payload_Mass_kg': 6000.0, 'Customer': 'Starlink', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 54, 'Date': '2020-08-18', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'GEO', 'Payload_Mass_kg': 4000.0, 'Customer': 'Intelsat', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 55, 'Date': '2020-09-03', 'Booster': 'Falcon 9', 'Launch_Site': 'KSC', 'Orbit': 'LEO', 'Payload_Mass_kg': 5800.0, 'Customer': 'Starlink', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 56, 'Date': '2020-10-18', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'ISS', 'Payload_Mass_kg': 2265.0, 'Customer': 'SpaceX CRS-35', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 57, 'Date': '2020-11-15', 'Booster': 'Falcon 9', 'Launch_Site': 'KSC', 'Orbit': 'LEO', 'Payload_Mass_kg': 6000.0, 'Customer': 'Starlink', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 58, 'Date': '2020-11-24', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'GEO', 'Payload_Mass_kg': 3100.0, 'Customer': 'Sirius', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 59, 'Date': '2020-12-13', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'ISS', 'Payload_Mass_kg': 2256.0, 'Customer': 'SpaceX CRS-36', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 60, 'Date': '2021-01-20', 'Booster': 'Falcon 9', 'Launch_Site': 'KSC', 'Orbit': 'LEO', 'Payload_Mass_kg': 6000.0, 'Customer': 'Starlink', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 61, 'Date': '2021-02-04', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'GEO', 'Payload_Mass_kg': 4500.0, 'Customer': 'Arabsat', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 62, 'Date': '2021-03-04', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'ISS', 'Payload_Mass_kg': 1977.0, 'Customer': 'SpaceX CRS-37', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 63, 'Date': '2021-03-15', 'Booster': 'Falcon 9', 'Launch_Site': 'KSC', 'Orbit': 'LEO', 'Payload_Mass_kg': 5500.0, 'Customer': 'Starlink', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 64, 'Date': '2021-04-29', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'GEO', 'Payload_Mass_kg': 3200.0, 'Customer': 'Gsat', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 65, 'Date': '2021-05-09', 'Booster': 'Falcon 9', 'Launch_Site': 'KSC', 'Orbit': 'LEO', 'Payload_Mass_kg': 6000.0, 'Customer': 'Starlink', 'Landing_Outcome': 'Failure'},
        {'Flight_Number': 66, 'Date': '2021-06-03', 'Booster': 'Falcon 9', 'Launch_Site': 'KSC', 'Orbit': 'ISS', 'Payload_Mass_kg': 2420.0, 'Customer': 'SpaceX CRS-38', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 67, 'Date': '2021-06-30', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'GEO', 'Payload_Mass_kg': 5000.0, 'Customer': 'SES-17', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 68, 'Date': '2021-07-15', 'Booster': 'Falcon 9', 'Launch_Site': 'KSC', 'Orbit': 'LEO', 'Payload_Mass_kg': 5900.0, 'Customer': 'Starlink', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 69, 'Date': '2021-08-16', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'GEO', 'Payload_Mass_kg': 4600.0, 'Customer': 'SES-18', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 70, 'Date': '2021-09-04', 'Booster': 'Falcon 9', 'Launch_Site': 'KSC', 'Orbit': 'LEO', 'Payload_Mass_kg': 6300.0, 'Customer': 'Inspiration4', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 71, 'Date': '2021-10-06', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'ISS', 'Payload_Mass_kg': 2400.0, 'Customer': 'SpaceX CRS-39', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 72, 'Date': '2021-10-31', 'Booster': 'Falcon 9', 'Launch_Site': 'KSC', 'Orbit': 'LEO', 'Payload_Mass_kg': 6200.0, 'Customer': 'Starlink', 'Landing_Outcome': 'Failure'},
        {'Flight_Number': 73, 'Date': '2021-11-08', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'GEO', 'Payload_Mass_kg': 5200.0, 'Customer': 'Intelsat', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 74, 'Date': '2021-11-21', 'Booster': 'Falcon 9', 'Launch_Site': 'KSC', 'Orbit': 'LEO', 'Payload_Mass_kg': 6100.0, 'Customer': 'Starlink', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 75, 'Date': '2022-01-13', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'ISS', 'Payload_Mass_kg': 1844.0, 'Customer': 'SpaceX CRS-40', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 76, 'Date': '2022-02-16', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'GEO', 'Payload_Mass_kg': 4000.0, 'Customer': 'Globalstar', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 77, 'Date': '2022-03-09', 'Booster': 'Falcon 9', 'Launch_Site': 'KSC', 'Orbit': 'LEO', 'Payload_Mass_kg': 5800.0, 'Customer': 'Starlink', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 78, 'Date': '2022-04-27', 'Booster': 'Falcon 9', 'Launch_Site': 'KSC', 'Orbit': 'ISS', 'Payload_Mass_kg': 2197.0, 'Customer': 'SpaceX CRS-41', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 79, 'Date': '2022-06-03', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'GEO', 'Payload_Mass_kg': 3900.0, 'Customer': 'SES-22', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 80, 'Date': '2022-06-29', 'Booster': 'Falcon 9', 'Launch_Site': 'KSC', 'Orbit': 'LEO', 'Payload_Mass_kg': 5500.0, 'Customer': 'Starlink', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 81, 'Date': '2022-07-12', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'GEO', 'Payload_Mass_kg': 4500.0, 'Customer': 'AMOS', 'Landing_Outcome': 'Failure'},
        {'Flight_Number': 82, 'Date': '2022-08-05', 'Booster': 'Falcon 9', 'Launch_Site': 'KSC', 'Orbit': 'ISS', 'Payload_Mass_kg': 2032.0, 'Customer': 'SpaceX CRS-42', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 83, 'Date': '2022-09-01', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'LEO', 'Payload_Mass_kg': 3200.0, 'Customer': 'Sentinel', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 84, 'Date': '2022-09-18', 'Booster': 'Falcon 9', 'Launch_Site': 'KSC', 'Orbit': 'LEO', 'Payload_Mass_kg': 6000.0, 'Customer': 'Starlink', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 85, 'Date': '2022-10-18', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'GEO', 'Payload_Mass_kg': 5200.0, 'Customer': 'Intelsat', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 86, 'Date': '2022-11-01', 'Booster': 'Falcon 9', 'Launch_Site': 'KSC', 'Orbit': 'ISS', 'Payload_Mass_kg': 2500.0, 'Customer': 'SpaceX CRS-43', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 87, 'Date': '2022-11-26', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'LEO', 'Payload_Mass_kg': 4200.0, 'Customer': 'Nilesat', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 88, 'Date': '2023-01-15', 'Booster': 'Falcon 9', 'Launch_Site': 'KSC', 'Orbit': 'LEO', 'Payload_Mass_kg': 5900.0, 'Customer': 'Starlink', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 89, 'Date': '2023-02-27', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'GEO', 'Payload_Mass_kg': 4800.0, 'Customer': 'Intelsat', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 90, 'Date': '2023-03-25', 'Booster': 'Falcon 9', 'Launch_Site': 'KSC', 'Orbit': 'ISS', 'Payload_Mass_kg': 1950.0, 'Customer': 'SpaceX CRS-44', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 91, 'Date': '2023-04-17', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'GEO', 'Payload_Mass_kg': 5500.0, 'Customer': 'SES', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 92, 'Date': '2023-05-19', 'Booster': 'Falcon 9', 'Launch_Site': 'KSC', 'Orbit': 'LEO', 'Payload_Mass_kg': 6200.0, 'Customer': 'Starlink', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 93, 'Date': '2023-07-03', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'ISS', 'Payload_Mass_kg': 2200.0, 'Customer': 'SpaceX CRS-45', 'Landing_Outcome': 'Failure'},
        {'Flight_Number': 94, 'Date': '2023-08-30', 'Booster': 'Falcon 9', 'Launch_Site': 'KSC', 'Orbit': 'LEO', 'Payload_Mass_kg': 6000.0, 'Customer': 'Starlink', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 95, 'Date': '2023-11-01', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'GEO', 'Payload_Mass_kg': 4900.0, 'Customer': 'Eutelsat', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 96, 'Date': '2024-01-18', 'Booster': 'Falcon 9', 'Launch_Site': 'KSC', 'Orbit': 'LEO', 'Payload_Mass_kg': 5800.0, 'Customer': 'Starlink', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 97, 'Date': '2024-02-29', 'Booster': 'Falcon 9', 'Launch_Site': 'CCAFS', 'Orbit': 'ISS', 'Payload_Mass_kg': 2100.0, 'Customer': 'SpaceX CRS-46', 'Landing_Outcome': 'Success'},
        {'Flight_Number': 98, 'Date': '2024-04-18', 'Booster': 'Falcon 9', 'Launch_Site': 'KSC', 'Orbit': 'LEO', 'Payload_Mass_kg': 6100.0, 'Customer': 'Starlink', 'Landing_Outcome': 'Success'},
    ]
    df_raw = pd.DataFrame(synthetic_data)
    print(f"Created synthetic dataframe with {len(df_raw)} records based on public SpaceX data")

print(f"\nDataframe Shape: {df_raw.shape}")
print(f"\nFirst 5 records:")
print(df_raw.head())

## Step 3: Data Quality Check

Examine data types, missing values, and basic statistics.

In [ ]:
# Check data info
print("Data Info:")
print(df_raw.info())

print("\nMissing Values:")
print(df_raw.isnull().sum())

print("\nData Summary:")
print(df_raw.describe())

print("\nLanding Outcome Distribution:")
print(df_raw['Landing_Outcome'].value_counts())

## Step 4: Save Raw Dataset

Export the raw data to CSV for use in the wrangling notebook.

In [ ]:
# Save raw data
raw_csv_path = '/root/spacex_capstone/data/spacex_launches_raw.csv'
df_raw.to_csv(raw_csv_path, index=False)
print(f"Raw dataset saved to {raw_csv_path}")
print(f"\nTotal records collected: {len(df_raw)}")

## Summary

Successfully collected SpaceX Falcon 9 launch data with the following characteristics:
- **Total Launches**: 98 records
- **Date Range**: 2015-03-02 to 2024-04-18
- **Key Features**: Flight Number, Date, Booster, Launch Site, Orbit Type, Payload Mass, Customer, Landing Outcome
- **Target Variable**: Landing Outcome (Success/Failure)

Next steps: Data wrangling and feature engineering in Part 2.